# Triple Extraction with spaCy: NLP-Based Knowledge Graph Construction

Traditional NLP pipelines extract knowledge graph triples through a series of stages:

1. **Named Entity Recognition (NER)** — detect entities (people, organizations, locations)
2. **Dependency Parsing** — analyze grammatical structure to find subject-verb-object patterns
3. **Relation Extraction** — identify relationships between entity pairs
4. **Triple Construction** — emit `(subject, predicate, object)` tuples

**Strengths**: Predictable, high precision in narrow domains, fast and cheap at scale.  
**Weaknesses**: Lower recall, requires domain-specific training, misses implicit relations.

In [ ]:
%pip install spacy networkx matplotlib -q

In [ ]:
import subprocess, sys

# Download spaCy model if not already available
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    import spacy
    nlp = spacy.load("en_core_web_sm")

import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print(f"spaCy version: {spacy.__version__}")
print(f"Model: en_core_web_sm")

## 1. Named Entity Recognition (NER)

NER detects spans of text that refer to real-world entities and classifies them into types like `PERSON`, `ORG`, `GPE` (geopolitical entity), `DATE`, etc.

In [ ]:
text = """Alan Turing worked at Bletchley Park during World War II. 
He was born in London in 1912. 
Turing developed the Turing machine concept at Cambridge University."""

doc = nlp(text)

print(f"{'Entity':<25} {'Label':<10} {'Description'}")
print("-" * 60)
for ent in doc.ents:
    print(f"{ent.text:<25} {ent.label_:<10} {spacy.explain(ent.label_)}")

## 2. Dependency Parsing for Relation Extraction

Dependency trees show grammatical relationships between words. We can extract **Subject-Verb-Object (SVO)** patterns by walking the parse tree to find:
- Verbs (ROOT or with relevant dependency tags)
- Their subjects (`nsubj`)
- Their objects (`dobj`, `pobj`, `attr`)

In [ ]:
# Visualize dependency parse for a simple sentence
simple_doc = nlp("Alan Turing worked at Bletchley Park.")

print(f"{'Token':<20} {'Dep':<12} {'Head':<20} {'POS'}")
print("-" * 60)
for token in simple_doc:
    print(f"{token.text:<20} {token.dep_:<12} {token.head.text:<20} {token.pos_}")

In [ ]:
def extract_svo_triples(doc):
    """Extract subject-verb-object triples from a spaCy Doc."""
    triples = []
    for token in doc:
        if token.pos_ == "VERB":
            # Find subject
            subjects = [child for child in token.children if child.dep_ in ("nsubj", "nsubjpass")]
            # Find objects (direct objects and prepositional objects)
            objects = []
            for child in token.children:
                if child.dep_ in ("dobj", "attr"):
                    objects.append(child)
                elif child.dep_ == "prep":
                    # Get the object of the preposition
                    for pobj in child.children:
                        if pobj.dep_ == "pobj":
                            objects.append(pobj)
            
            # Build triples from all subject-object combinations
            for subj in subjects:
                subj_text = _get_compound(subj)
                for obj in objects:
                    obj_text = _get_compound(obj)
                    # Include preposition in the verb if present
                    verb_text = token.lemma_
                    prep = [c for c in token.children if c.dep_ == "prep" and obj in c.subtree]
                    if prep:
                        verb_text = f"{token.lemma_}_{prep[0].text}"
                    triples.append((subj_text, verb_text, obj_text))
    return triples

def _get_compound(token):
    """Get the full compound noun phrase for a token."""
    compounds = [child.text for child in token.lefts if child.dep_ == "compound"]
    return " ".join(compounds + [token.text])

# Test on multiple sentences
test_text = """Alan Turing worked at Bletchley Park. 
Marie Curie discovered radium. 
Einstein developed the theory of relativity."""

test_doc = nlp(test_text)
svo_triples = extract_svo_triples(test_doc)

print("Extracted SVO triples:\n")
for s, v, o in svo_triples:
    print(f"  ({s}) —{v}→ ({o})")

## 3. Entity-Aware Triple Extraction

The next step is combining NER with dependency parsing: extract **relations between named entities** and tag each entity with its type.

In [ ]:
ai_text = """Geoffrey Hinton joined Google Brain in 2013. 
Yann LeCun leads the AI research division at Meta. 
Yoshua Bengio co-founded Mila in Montreal. 
Hinton and LeCun both received the Turing Award in 2018. 
Demis Hassabis founded DeepMind in London."""

ai_doc = nlp(ai_text)

# First, show all entities found
print("Named entities detected:\n")
print(f"{'Entity':<25} {'Label':<10}")
print("-" * 35)
for ent in ai_doc.ents:
    print(f"{ent.text:<25} {ent.label_}")

In [ ]:
def extract_entity_triples(doc):
    """Extract triples where subject and object are named entities."""
    triples = []
    # Build entity lookup: token index → entity
    token_to_ent = {}
    for ent in doc.ents:
        for token in ent:
            token_to_ent[token.i] = ent
    
    for token in doc:
        if token.pos_ == "VERB":
            # Find subject entities
            subjects = []
            for child in token.children:
                if child.dep_ in ("nsubj", "nsubjpass"):
                    if child.i in token_to_ent:
                        subjects.append(token_to_ent[child.i])
                    else:
                        # Check compound tokens
                        for c in child.subtree:
                            if c.i in token_to_ent:
                                subjects.append(token_to_ent[c.i])
                                break
            
            # Find object entities
            objects = []
            for child in token.children:
                if child.dep_ in ("dobj", "attr"):
                    for c in child.subtree:
                        if c.i in token_to_ent:
                            objects.append(token_to_ent[c.i])
                elif child.dep_ == "prep":
                    for c in child.subtree:
                        if c.i in token_to_ent:
                            objects.append(token_to_ent[c.i])
            
            # Build verb phrase
            verb = token.lemma_
            preps = [c for c in token.children if c.dep_ == "prep"]
            if preps:
                verb = f"{verb}_{preps[0].text}"
            
            for subj in subjects:
                for obj in objects:
                    if subj != obj:
                        triples.append({
                            "subject": subj.text,
                            "subject_type": subj.label_,
                            "predicate": verb,
                            "object": obj.text,
                            "object_type": obj.label_
                        })
    return triples

entity_triples = extract_entity_triples(ai_doc)

print("Entity-aware triples:\n")
for t in entity_triples:
    print(f"  ({t['subject']} [{t['subject_type']}]) —{t['predicate']}→ ({t['object']} [{t['object_type']}])")

## 4. Building a Knowledge Graph from Extracted Triples

Now we combine all extracted triples into a NetworkX graph and visualize the result.

In [ ]:
G_kg = nx.DiGraph()

# Collect node types for coloring
node_types = {}

for t in entity_triples:
    G_kg.add_edge(t["subject"], t["object"], relation=t["predicate"])
    node_types[t["subject"]] = t["subject_type"]
    node_types[t["object"]] = t["object_type"]

# Also add SVO triples from the AI text for more coverage
svo = extract_svo_triples(ai_doc)
for s, v, o in svo:
    if s not in G_kg.nodes() and o not in G_kg.nodes():
        continue  # Only add if at least one node already exists
    G_kg.add_edge(s, o, relation=v)

# Color by entity type
type_colors = {"PERSON": "#FF9999", "ORG": "#99CCFF", "GPE": "#99FF99", 
               "DATE": "#FFCC99", "WORK_OF_ART": "#CC99FF"}
color_map = [type_colors.get(node_types.get(n, ""), "#D9D9D9") for n in G_kg.nodes()]

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G_kg, seed=42, k=2)
nx.draw(G_kg, pos, with_labels=True, node_color=color_map,
        node_size=2500, font_size=9, font_weight="bold",
        arrows=True, arrowsize=15, edge_color="gray")

edge_labels = nx.get_edge_attributes(G_kg, "relation")
nx.draw_networkx_edge_labels(G_kg, pos, edge_labels=edge_labels, font_size=8)

legend_handles = [mpatches.Patch(color=c, label=t) for t, c in type_colors.items()]
legend_handles.append(mpatches.Patch(color="#D9D9D9", label="Other"))
plt.legend(handles=legend_handles, loc="upper left", fontsize=10)
plt.title("Knowledge Graph Extracted with spaCy", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nGraph: {G_kg.number_of_nodes()} nodes, {G_kg.number_of_edges()} edges")

## 5. Pattern-Based Relation Extraction

A more robust approach uses **spaCy's Matcher** to define specific patterns for relations. This gives more control over what gets extracted.

In [ ]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)

# Pattern: PERSON + "founded" + ORG
# We match the verb patterns and then look for entities around them
founded_pattern = [{"LEMMA": {"IN": ["found", "co-found", "establish", "create"]}}]
joined_pattern = [{"LEMMA": {"IN": ["join", "work"]}}]
leads_pattern = [{"LEMMA": {"IN": ["lead", "head", "direct", "manage"]}}]
received_pattern = [{"LEMMA": {"IN": ["receive", "win", "earn", "get"]}}]

matcher.add("FOUNDED", [founded_pattern])
matcher.add("JOINED", [joined_pattern])
matcher.add("LEADS", [leads_pattern])
matcher.add("RECEIVED", [received_pattern])

def extract_pattern_triples(doc, matcher):
    """Use matcher patterns to find relation verbs, then link surrounding entities."""
    matches = matcher(doc)
    triples = []
    
    for match_id, start, end in matches:
        rule_name = nlp.vocab.strings[match_id]
        verb_token = doc[start]
        sent = verb_token.sent
        
        # Get entities in the same sentence
        sent_ents = [ent for ent in doc.ents if ent.start >= sent.start and ent.end <= sent.end]
        
        # Heuristic: first PERSON entity is subject, first ORG/GPE is object
        persons = [e for e in sent_ents if e.label_ in ("PERSON",)]
        orgs = [e for e in sent_ents if e.label_ in ("ORG", "GPE", "WORK_OF_ART")]
        
        for person in persons:
            for org in orgs:
                relation = rule_name.lower()
                triples.append({
                    "subject": person.text,
                    "predicate": relation,
                    "object": org.text
                })
    
    # Deduplicate
    seen = set()
    unique = []
    for t in triples:
        key = (t["subject"], t["predicate"], t["object"])
        if key not in seen:
            seen.add(key)
            unique.append(t)
    return unique

pattern_triples = extract_pattern_triples(ai_doc, matcher)

print("Pattern-based triples:\n")
for t in pattern_triples:
    print(f"  ({t['subject']}) —{t['predicate']}→ ({t['object']})")

In [ ]:
# Compare: SVO extraction vs Pattern-based extraction
print("=== Comparison ===")
print(f"\nSVO-based triples: {len(entity_triples)}")
for t in entity_triples:
    print(f"  ({t['subject']}) —{t['predicate']}→ ({t['object']})")

print(f"\nPattern-based triples: {len(pattern_triples)}")
for t in pattern_triples:
    print(f"  ({t['subject']}) —{t['predicate']}→ ({t['object']})")

print("\n--- Notes ---")
print("SVO extraction captures more diverse relations but may include noise.")
print("Pattern-based extraction is more precise but only finds pre-defined relation types.")

## Limitations and When to Use spaCy

| Aspect | spaCy NLP Pipeline |
|--------|-------------------|
| **Precision** | High for well-defined patterns |
| **Recall** | Lower — misses implicit and paraphrased relations |
| **Speed** | Fast, CPU-friendly, scales to large corpora |
| **Setup cost** | Requires domain-specific NER/RE training for best results |
| **Best for** | Recurring patterns, narrow domains, batch processing |

**Key insight**: For agentic systems, a **hybrid approach** often works best:
- Use spaCy for entity detection (fast, reliable)
- Use an LLM for relation extraction (flexible, high recall)
- Validate extracted triples with rules (check entities appear in source text)

See **Notebook 04** for LLM-based triple extraction and comparison.